# Cobblemon Atlas｜動作真實性稽核（輕量版）

直接讀取前一輪已存在的 `resolved-render-pilot.json`，不重新解壓資源包、不重新渲染 24 幀動畫。

檢查同片段共用、動作場景不一致、靜態姿態；報告保留在 `Cobblemon_Atlas/reports/resolver-validation/render-pilot/semantic-audit.json`。任何候選都不直接發佈到網站。


In [ ]:
from pathlib import Path
import subprocess,sys,json
from google.colab import drive
drive.mount('/content/drive')
root=Path('/content/drive/MyDrive/Cobblemon_Atlas')
report=root/'reports/resolver-validation/render-pilot/resolved-render-pilot.json'
assert report.is_file(),f'找不到既有測試報告：{report}'
print('Found render report:',report)


In [ ]:
repo=Path('/content/cobblemon-atlas-semantic-audit')
branch='feat/animation-manifest-scanner-phase1'
if not (repo/'.git').is_dir():
    subprocess.run(['git','clone','--depth','1','--filter=blob:none','--sparse','--branch',branch,
                    'https://github.com/hwh0410/cobblemon-atlas.git',str(repo)],check=True)
subprocess.run(['git','-C',str(repo),'sparse-checkout','set','tools'],check=True)
subprocess.run(['git','-C',str(repo),'pull','--ff-only','origin',branch],check=True)
out=report.parent/'semantic-audit.json'
subprocess.run([sys.executable,str(repo/'tools/audit_pose_semantics.py'),
                '--report',str(report),'--out',str(out)],check=True)
result=json.loads(out.read_text(encoding='utf-8'))
print('Summary:',result['summary'])
print('Saved:',out)
